In [6]:
import os
import random
import json
from functools import partial

import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torch.optim import Adam
import numpy as np
import json
from  models.accession_dataset import AccessionDataset, ocular_collate_fn
from models.lstm import OcularStatefulLSTM

SELECTED_FREQUENCY = '0.75'
DATASET_PATH = '../exports/dataset'
RESULTS_DIR = '../exports'

SAMPLING_RATE = 240

FREQUENCIES_WINDOWS = {
    "0.5": int(SAMPLING_RATE * 2),
    "0.75": int(SAMPLING_RATE * 1.5),
    "1.0": int(SAMPLING_RATE * 1.0),
}

window_size = FREQUENCIES_WINDOWS[SELECTED_FREQUENCY]

In [4]:
mg_samples = os.listdir(os.path.join(DATASET_PATH, "Definite MG"))
healthy_samples = os.listdir(os.path.join(DATASET_PATH, "Healthy control"))

print("MG len", len(mg_samples))
print("Healthy len", len(healthy_samples))

MG len 63
Healthy len 85


# 5-fold cross validation

We are generating crosss validation to check the performance of our model because dataset is short

In [7]:
seed = 42
random.seed(seed)

n_folds = 5

healthy_shuffled = healthy_samples.copy()
mg_shuffled = mg_samples.copy()

random.shuffle(healthy_shuffled)
random.shuffle(mg_shuffled)

folds = []
for fold_idx in range(n_folds):
    val_healthy = healthy_shuffled[fold_idx::n_folds]
    train_healthy = [s for i, s in enumerate(healthy_shuffled) if i % n_folds != fold_idx]

    val_mg = mg_shuffled[fold_idx::n_folds]
    train_mg = [s for i, s in enumerate(mg_shuffled) if i % n_folds != fold_idx]

    folds.append({
        "fold": fold_idx,
        "train": {
            "healthy": train_healthy,
            "mg": train_mg,
        },
        "val": {
            "healthy": val_healthy,
            "mg": val_mg,
        },
    })

config = {
    "seed": seed,
    "n_folds": n_folds,
    "dataset_path": DATASET_PATH,
    "folds": folds,
}

with open(os.path.join(RESULTS_DIR, "cv_config.json"), "w") as f:
    json.dump(config, f, indent=2)

print("Configuration of 5-fold saved in cv_config.json")

Configuration of 5-fold saved in cv_config.json


In [8]:
def run_epoch(model, loader, optimizer, criterion, device, is_train=True):
    if is_train:
        model.train()
        torch.set_grad_enabled(True)
    else:
        model.eval()
        torch.set_grad_enabled(False)
        
    total_loss = 0.0
    correct = 0
    total = 0
    
    for sequences, labels, masks in loader:
        sequences = sequences.to(device)
        labels = labels.to(device).long()
        masks = masks.to(device)
        
        outputs = model(sequences, masks)
        loss = criterion(outputs, labels)
        
        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        total_loss += loss.item() * labels.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    return total_loss / total, 100 * correct / total

In [ ]:
def train_with_json_splits(splits_json_path, base_dataset_dir, epochs=15, batch_size=1):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Load your custom split configurations
    with open(splits_json_path, 'r') as f:
        config = json.load(f)
        
    folds_config = config.get('folds', [])
    print(f"Loaded config containing {len(folds_config)} pre-configured folds.")

    fold_train_accs, fold_val_accs = [], []

    # Iterate through each predefined fold in the JSON
    for fold_data in folds_config:
        fold_idx = fold_data['fold']
        print(f"\n" + "="*50)
        print(f" TRAINING PREDEFINED FOLD {fold_idx + 1}/{len(folds_config)} ")
        print("="*50)

        # Helper function to construct absolute/relative paths from raw filenames
        def resolve_paths(split_section):
            paths = []
            # Check Healthy files under 'Healthy control' folder
            for f_name in split_section.get('healthy', []):
                paths.append(os.path.join(base_dataset_dir, 'Healthy control', f_name))
            # Check MG files under 'Definite MG' and 'Probable MG' subfolders
            for f_name in split_section.get('mg', []):
                definite_path = os.path.join(base_dataset_dir, 'Definite MG', f_name)
                probable_path = os.path.join(base_dataset_dir, 'Probable MG', f_name)
                if os.path.exists(definite_path):
                    paths.append(definite_path)
                elif os.path.exists(probable_path):
                    paths.append(probable_path)
                else:
                    # Fallback if your export directory tree doesn't use subfolders
                    paths.append(os.path.join(base_dataset_dir, f_name))
            return paths

        # Resolve complete file structures for this split
        train_paths = resolve_paths(fold_data['train'])
        val_paths = resolve_paths(fold_data['val'])

        # Initialize PyTorch Datasets
        train_dataset = AccessionDataset(train_paths, window_size=window_size, frequency_key='freq_1.0', )
        val_dataset = AccessionDataset(val_paths, window_size=window_size, frequency_key='freq_1.0')

        # Build active DataLoaders using your customized collate mechanism
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=partial(ocular_collate_fn, window_size=window_size))
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=partial(ocular_collate_fn, window_size=window_size))

        # Initialize fresh binary classification model parameters
        model = OcularStatefulLSTM(num_classes=2).to(device)
        criterion = nn.CrossEntropyLoss()
        # optimizer = Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
        optimizer = Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

        best_val_acc = 0.0

        for epoch in range(epochs):
            train_loss, train_acc = run_epoch(model, train_loader, optimizer, criterion, device, is_train=True)
            val_loss, val_acc = run_epoch(model, val_loader, None, criterion, device, is_train=False)
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                
            print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%")

        print(f">> Fold {fold_idx + 1} Done. Best Validation Accuracy: {best_val_acc:.2f}%")
        fold_val_accs.append(best_val_acc)
        fold_train_accs.append(train_acc)

    print("\n" + "═"*50)
    print(" FINAL PREDEFINED CROSS VALIDATION METRICS SUMMARY")
    print("═"*50)
    print(f"Mean Training Accuracy: {np.mean(fold_train_accs):.2f}%")
    print(f"Mean Validation Accuracy: {np.mean(fold_val_accs):.2f}% (+/- {np.std(fold_val_accs):.2f}%)")

In [10]:
# Set paths pointing to your local assets
SPLITS_JSON = os.path.join(RESULTS_DIR, "cv_config.json")          # Path to your split manifest json

train_with_json_splits(SPLITS_JSON, DATASET_PATH, epochs=15, batch_size=8)

Using device: cpu
Loaded config containing 5 pre-configured folds.

 TRAINING PREDEFINED FOLD 1/5 
[]
[]


ValueError: num_samples should be a positive integer value, but got num_samples=0